# Data Governance and Privacy 




# 1. Audit Data

## 1. Load Data

In [37]:
import pandas as pd
import json

# Load raw 
with open("../data/raw_credit_applications.json") as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.shape

(502, 21)

## 2. Data Quality Dimensions

The audit evaluates the following six dimensions:

1. **Accuracy** — Do values reflect the real-world truth?
2. **Completeness** — Are all required fields populated?
3. **Consistency** — Are values coherent across records?
4. **Timeliness** — Is the data sufficiently up to date?
5. **Validity** — Do values fall within acceptable ranges or formats?
6. **Uniqueness** — Are records free from unwanted duplicates?

## 3. Audit Query 1 — Uniqueness

**Purpose:** Identify duplicate values in fields that should be unique (`_id`, `ssn`, `email`).

In [38]:
# Uniqueness: Check duplicates on fields that should be unique
for col in ["_id", "applicant_info.ssn", "applicant_info.email"]:
    dups = df[df[col].notna() & df.duplicated(subset=[col], keep=False)]
    print(f"Duplicate {col}: {len(dups)} records")

Duplicate _id: 4 records
Duplicate applicant_info.ssn: 6 records
Duplicate applicant_info.email: 11 records


## 4. Audit Query 2 — Completeness

**Purpose:** Identify missing or blank values across all fields. Structural nulls (e.g., `interest_rate`/`approved_amount` for denied applications, `rejection_reason` for approved ones) are excluded.

In [39]:
# Completeness: Missing values per column
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

for col, count in missing.items():
    pct = count / len(df) * 100
    print(f"  {col}: {count} ({pct:.1f}%)")

# Check blank strings in date_of_birth (not captured by isna)
blank_dateofbirth = (df["applicant_info.date_of_birth"] == "").sum()
print(f"\n  applicant_info.date_of_birth (blank strings): {blank_dateofbirth}")


  notes: 500 (99.6%)
  financials.annual_salary: 497 (99.0%)
  loan_purpose: 452 (90.0%)
  processing_timestamp: 440 (87.6%)
  decision.rejection_reason: 292 (58.2%)
  decision.approved_amount: 210 (41.8%)
  decision.interest_rate: 210 (41.8%)
  applicant_info.ip_address: 5 (1.0%)
  applicant_info.ssn: 5 (1.0%)
  financials.annual_income: 5 (1.0%)
  applicant_info.gender: 1 (0.2%)
  applicant_info.date_of_birth: 1 (0.2%)
  applicant_info.zip_code: 1 (0.2%)

  applicant_info.date_of_birth (blank strings): 4


## 5. Audit Query 3 — Consistency

**Purpose:** Detect inconsistencies in categorical encodings and logical contradictions between fields.

In [40]:
# Consistency: Gender encoding
print("=== Gender distinct values ===")
print(df["applicant_info.gender"].value_counts(dropna=False).to_string())

#
dob = df["applicant_info.date_of_birth"].dropna()
dob = dob[dob != ""]
iso_mask = dob.str.match(r"^\d{4}-\d{2}-\d{2}$")
print(f"\n=== Date of birth formats ===")
print(f"ISO (YYYY-MM-DD): {iso_mask.sum()}")
print(f"Non-ISO formats:  {(~iso_mask).sum()}")

# Consistency: annual_income stored as string instead of numeric
print(f"\n=== annual_income dtype ===")
print(f"Stored as: {df['financials.annual_income'].dtype} (expected numeric)")

# Consistency: loan_approved vs decision fields
approved = df[df["decision.loan_approved"] == True]
denied = df[df["decision.loan_approved"] == False]
print("\n=== Decision field consistency ===")
print(f"Approved with no interest_rate: {approved['decision.interest_rate'].isna().sum()}")
print(f"Approved with no approved_amount: {approved['decision.approved_amount'].isna().sum()}")
print(f"Denied with interest_rate set: {denied['decision.interest_rate'].notna().sum()}")
print(f"Denied with approved_amount set: {denied['decision.approved_amount'].notna().sum()}")
print(f"Approved with rejection_reason: {approved['decision.rejection_reason'].notna().sum()}")
print(f"Denied with no rejection_reason: {denied['decision.rejection_reason'].isna().sum()}")

=== Gender distinct values ===
applicant_info.gender
Male      195
Female    193
F          58
M          53
            2
NaN         1

=== Date of birth formats ===
ISO (YYYY-MM-DD): 340
Non-ISO formats:  157

=== annual_income dtype ===
Stored as: object (expected numeric)

=== Decision field consistency ===
Approved with no interest_rate: 0
Approved with no approved_amount: 0
Denied with interest_rate set: 0
Denied with approved_amount set: 0
Approved with rejection_reason: 0
Denied with no rejection_reason: 0


## 6. Audit Query 4 — Validity

**Purpose:** Detect values outside acceptable ranges or with invalid formats.

In [41]:
import re

# Convert annual_income to numeric for range checks
income = pd.to_numeric(df["financials.annual_income"], errors="coerce")
#RAnge checks for financial fields
print(f"annual_income == 0:        {(income == 0).sum()} record(s)")
print(f"annual_income < 0:         {(income < 0).sum()} record(s)")
print(f"debt_to_income > 1:        {(df['financials.debt_to_income'] > 1).sum()} record(s)")
print(f"debt_to_income < 0:        {(df['financials.debt_to_income'] < 0).sum()} record(s)")
print(f"credit_history_months < 0: {(df['financials.credit_history_months'] < 0).sum()} record(s)")
print(f"savings_balance < 0:       {(df['financials.savings_balance'] < 0).sum()} record(s)")

print("\nEmail format check")
email_format = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
em = df["applicant_info.email"]
mask = em.notna()
invalid_emails = df[mask & em[mask].apply(lambda x: not bool(re.match(email_format, x)))]
print(f"Invalid email format: {len(invalid_emails)} record(s)")
if len(invalid_emails) > 0:
    print(invalid_emails[["_id", "applicant_info.email"]].to_string(index=False))

annual_income == 0:        1 record(s)
annual_income < 0:         0 record(s)
debt_to_income > 1:        1 record(s)
debt_to_income < 0:        0 record(s)
credit_history_months < 0: 2 record(s)
savings_balance < 0:       1 record(s)

Email format check
Invalid email format: 11 record(s)
    _id   applicant_info.email
app_075                       
app_204 mike johnson@gmail.com
app_299  test.user.outlook.com
app_413                       
app_120                       
app_068       john.doe@invalid
app_268                       
app_377                       
app_146           sarah.smith@
app_350                       
app_165                       


## 7. Audit Summary

| Dimension | Check | Field(s) | Records affected | Governance relevance |
|-----------|-------|----------|------------------|----------------------|
| Uniqueness | Duplicate identifiers | `_id` | 4 (2 duplicate IDs) | Duplicate primary keys compromise record integrity |
| Uniqueness | Duplicate identifiers | `applicant_info.ssn` | 6 (3 duplicate SSNs) | May cause repeated processing or identity conflicts |
| Uniqueness | Duplicate identifiers | `applicant_info.email` | 11 (includes blanks treated as duplicates) | Blank emails collapse into false duplicates |
| Completeness | Missing values | `processing_timestamp` | 440 (87.6%) | Critical audit trail gap |
| Completeness | Missing values | `loan_purpose` | 452 (90.0%) | Limits ability to assess decision context |
| Completeness | Missing values | `applicant_info.ssn`, `applicant_info.ip_address`, `financials.annual_income` | 5 each | Core applicant/financial data incomplete |
| Completeness | Missing values | `applicant_info.gender`, `applicant_info.date_of_birth`, `applicant_info.zip_code` | 1 each | Minor demographic gaps |
| Completeness | Blank strings | `applicant_info.date_of_birth` | 4 | Blank strings not captured by `isna()` — hidden missingness |
| Completeness | Structural nulls (not flagged) | `decision.interest_rate`, `decision.approved_amount` / `decision.rejection_reason` | 210 / 292 | Expected pattern based on `decision.loan_approved` |
| Consistency | Gender encoded inconsistently | `applicant_info.gender` | 114 (M/F vs Male/Female + blank) | Mixed encodings affect reliability and downstream analysis |
| Consistency | Date of birth in mixed formats | `applicant_info.date_of_birth` | 161 non-ISO | Multiple formats (DD/MM/YYYY, YYYY/MM/DD) complicate parsing |
| Consistency | Annual income stored as string | `financials.annual_income` | All (dtype: object) | Type mismatch may cause silent errors in computations |
| Consistency | Decision field coherence | `decision.loan_approved` vs decision fields | 0 | All decision fields are logically consistent |
| Validity | Zero annual income | `financials.annual_income` | 1 | May indicate data entry error |
| Validity | Negative credit history months | `financials.credit_history_months` | 2 (values: −10, −3) | Impossible values — pipeline or entry error |
| Validity | Debt-to-income > 1 | `financials.debt_to_income` | 1 | Out-of-range ratio may distort risk assessment |
| Validity | Negative savings balance | `financials.savings_balance` | 1 (value: −5000) | Unexpected negative balance |
| Validity | Invalid email format | `applicant_info.email` | 11 (spaces, missing domain, blank) | Affects data usability and contact reliability |


# 2. PII Identification

Identify personal data fields present in the dataset and classifies them by identifier type and risk level.

## 2.1 Field Inspection

List all fields in the dataset to identify which ones contain personal data.

In [42]:

print(f"Total columns: {len(df.columns)}\n")
for col in df.columns:
    print(f"  {col} ({df[col].dtype})")

Total columns: 21

  _id (object)
  spending_behavior (object)
  processing_timestamp (object)
  applicant_info.full_name (object)
  applicant_info.email (object)
  applicant_info.ssn (object)
  applicant_info.ip_address (object)
  applicant_info.gender (object)
  applicant_info.date_of_birth (object)
  applicant_info.zip_code (object)
  financials.annual_income (object)
  financials.credit_history_months (int64)
  financials.debt_to_income (float64)
  financials.savings_balance (int64)
  decision.loan_approved (bool)
  decision.rejection_reason (object)
  loan_purpose (object)
  decision.interest_rate (float64)
  decision.approved_amount (float64)
  financials.annual_salary (float64)
  notes (object)


## 2.2 Direct Identifiers

Fields that can uniquely identify a person on their own.

In [43]:
# Direct identifiers: sample values
direct_ids = ["applicant_info.full_name", "applicant_info.ssn", "applicant_info.email", "applicant_info.ip_address"]
for col in direct_ids:
    print(f"{col}: {df[col].dropna().nunique()} unique values")
    print(f"  e.g. {df[col].dropna().iloc[0]}\n")

applicant_info.full_name: 475 unique values
  e.g. Jerry Smith

applicant_info.ssn: 494 unique values
  e.g. 596-64-4340

applicant_info.email: 494 unique values
  e.g. jerry.smith17@hotmail.com

applicant_info.ip_address: 496 unique values
  e.g. 192.168.48.155



## 2.3 Indirect Identifiers (Quasi-identifiers)

Fields that, when combined, may enable re-identification, even after removing direct identifiers.

> Sweeney (2000): 87% of Americans can be uniquely identified with just **ZIP code + gender + birth date**.

In [44]:
# Quasi-identifiers: check uniqueness of combinations
quasi_ids = ["applicant_info.gender", "applicant_info.date_of_birth", "applicant_info.zip_code"]
combo = df[quasi_ids].dropna()
unique_combos = combo.drop_duplicates().shape[0]
print(f"Unique combinations of {quasi_ids}: {unique_combos} out of {len(combo)} records")
print(f"Potentially re-identifiable: {unique_combos == len(combo)}")

Unique combinations of ['applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code']: 500 out of 501 records
Potentially re-identifiable: False


## 2.4 Sensitive / Higher-Risk Personal Data

Fields that may require stronger protection due to their nature.

In [45]:
# Sensitive data: fields with financial or behavioural information
sensitive_fields = [
    "financials.annual_income", "financials.debt_to_income",
    "financials.savings_balance", "financials.credit_history_months",
    "spending_behavior", "decision.loan_approved", "decision.rejection_reason"
]
for col in sensitive_fields:
    non_null = df[col].notna().sum()
    print(f"{col}: {non_null} non-null values")

financials.annual_income: 497 non-null values
financials.debt_to_income: 502 non-null values
financials.savings_balance: 502 non-null values
financials.credit_history_months: 502 non-null values
spending_behavior: 502 non-null values
decision.loan_approved: 502 non-null values
decision.rejection_reason: 210 non-null values


## 2.5 PII Classification Summary

| Field | Category | Identifier Type | Risk Note |
|-------|----------|----------------|-----------|
| `applicant_info.full_name` | Personal | Direct | Uniquely identifies a person |
| `applicant_info.ssn` | Personal | Direct | National identifier — high sensitivity |
| `applicant_info.email` | Personal | Direct | Uniquely identifies a person |
| `applicant_info.ip_address` | Personal | Direct | Can be linked to a person or location |
| `applicant_info.gender` | Demographic | Quasi-identifier | Re-identification risk when combined |
| `applicant_info.date_of_birth` | Demographic | Quasi-identifier | Re-identification risk when combined |
| `applicant_info.zip_code` | Demographic | Quasi-identifier | Re-identification risk when combined |
| `financials.annual_income` | Financial | Sensitive | Reveals economic status |
| `financials.debt_to_income` | Financial | Sensitive | Reveals financial risk profile |
| `financials.savings_balance` | Financial | Sensitive | Reveals economic status |
| `financials.credit_history_months` | Financial | Sensitive | Reveals financial history |
| `spending_behavior` | Behavioural | Sensitive | Reveals personal spending patterns |
| `decision.loan_approved` | Decision | Sensitive | Automated decision output |
| `decision.rejection_reason` | Decision | Sensitive | Reveals basis of automated decision |

# 3. Privacy information/data

| | **Pseudonymization** | **Anonymization** | **Differential Privacy** |
|---|---|---|---|
| **How it works** | Swap identifiers with tokens | Generalise or suppress values | Add calibrated noise to outputs |
| **Example** | John Smith → User_7834 | Age 35 → 30–40 | Avg salary ± random noise |
| **Reversible?** | Yes (with key) | No (if done right) | No (mathematical) |
| **Re-id risk** | Medium | Low | Very Low |
| **Data utility** | High | Medium | Lower |
| **GDPR status** | Still personal data | Not personal data | Not personal data |

## 3.1 Pseudonymization

Replace direct identifiers with pseudonyms. Still **personal data under GDPR** — it can be reversed with the mapping key (GDPR Article 4(5)).

Three main approaches from the slides:

| Technique | How it works | Risk |
|---|---|---|
| **Tokenization** | Random token + lookup table | Table stolen → all exposed |
| **Hashing (SHA-256)** | One-way function, same input → same hash | Deterministic → brute-force |
| **Encryption** | Secret key, reversible | Key compromise → full reversal |

**Important:** Pseudonymization alone is NOT anonymization.

### Hashing with SHA-256

Same input always gives the same hash, this is the problem. An attacker can try all common emails and match.

In [46]:
import hashlib


email = "john.smith@company.com"
hashed = hashlib.sha256(email.encode()).hexdigest()

print("Original:", email)
print("Hashed:  ", hashed)


print("\nSame hash?", hashed == hashlib.sha256(email.encode()).hexdigest())

Original: john.smith@company.com
Hashed:   fdc2a4abc53872d0f934b5614ddc312daa165895065bb00a5986849aade8c322

Same hash? True


### The fix: salting

Adding a secret random string (salt) before hashing makes brute-force infeasible.


In [47]:

salt = "x7Km9pQr"  

hashed_with_salt = hashlib.sha256((salt + email).encode()).hexdigest()

print("Without salt:", hashed)
print("With salt:   ", hashed_with_salt)
print("\nCompletely different! The salt makes pre-computation impossible.")

Without salt: fdc2a4abc53872d0f934b5614ddc312daa165895065bb00a5986849aade8c322
With salt:    78a3591a1003af3e99af1520e423c2cf7dc01c9a2bb726c7681b12e3b92933bc

Completely different! The salt makes pre-computation impossible.


### Applying pseudonymization to our dataset

We use **tokenization** for names, **salted hashing** for email/SSN, and **drop** the IP address entirely.

In [48]:
# Apply pseudonymization 
df_pseudo = df.copy()


import uuid
names = df_pseudo['applicant_info.full_name'].dropna().unique()
token_map = {name: f"USR_{i:04d}" for i, name in enumerate(names, 1)}
df_pseudo['applicant_info.full_name'] = df_pseudo['applicant_info.full_name'].map(token_map)

# Salted hashing on email and SSN
salt = "x7Km9pQr"
for col in ['applicant_info.email', 'applicant_info.ssn']:
    df_pseudo[col] = df_pseudo[col].apply(
        lambda x: hashlib.sha256((salt + str(x)).encode()).hexdigest()[:16] if pd.notna(x) else x
    )

# Drop IP address — direct identifier, not useful for analysis
df_pseudo = df_pseudo.drop(columns=['applicant_info.ip_address'])

# compare before and after
print("BEFORE:")
print(df[['applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn']].head(3).to_string())
print("\nAFTER:")
print(df_pseudo[['applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn']].head(3).to_string())

BEFORE:
  applicant_info.full_name       applicant_info.email applicant_info.ssn
0              Jerry Smith  jerry.smith17@hotmail.com        596-64-4340
1           Brandon Walker  brandon.walker2@yahoo.com        425-69-4784
2              Scott Moore     scott.moore94@mail.com        370-78-5178

AFTER:
  applicant_info.full_name applicant_info.email applicant_info.ssn
0                 USR_0001     aa3e580fcd04b4d9   0b9028ba19acfed4
1                 USR_0002     265658e9f71ef347   ab5de87b02695c33
2                 USR_0003     71abfd7702b1fd04   6e0ca7ca8330acc6


But this is **not enough** — quasi-identifiers like gender, DOB and zip code are still there. If a combination is unique, the person can be re-identified. We need to go further.

## 3.2 Anonymization — k-Anonymity

The idea: generalise or suppress quasi-identifiers so that every person looks like at least **k − 1** others.

- **Generalisation** — make values less specific (e.g. age 35 → 30–40)
- **Suppression** — remove values that are too unique

If done right, the data is **no longer personal data** under GDPR (Covered in the following section).

In [49]:
# check k-anonymity BEFORE generalisation
quasi_ids = ['applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code']

groups = df_pseudo.groupby(quasi_ids, dropna=False).size()

print("Current k-anonymity:", groups.min())
print("Unique individuals (k=1):", (groups == 1).sum())
print("Groups with k < 5:", (groups < 5).sum())

Current k-anonymity: 1
Unique individuals (k=1): 500
Groups with k < 5: 501


### Generalising our quasi-identifiers

- **Date of birth** → birth year only
- **Zip code** → first 3 digits
- **Gender** → standardise to M/F

In [ ]:
df_anon = df_pseudo.copy()

# date of birth → keep only the year
dob = pd.to_datetime(df_anon['applicant_info.date_of_birth'], errors='coerce')
df_anon['applicant_info.date_of_birth'] = dob.dt.year.astype('Int64')

# zip code → keep only first 3 digits
df_anon['applicant_info.zip_code'] = df_anon['applicant_info.zip_code'].astype(str).str[:3]

# standardise gender (M/F/Male/Female → M/F)
df_anon['applicant_info.gender'] = df_anon['applicant_info.gender'].str.upper().str[0]

# check k-anonymity AFTER generalisation
groups_after = df_anon.groupby(quasi_ids, dropna=False).size()

print("AFTER generalisation:")
print("k-anonymity:", groups_after.min())
print("Unique individuals (k=1):", (groups_after == 1).sum())
print("Groups with k < 5:", (groups_after < 5).sum())

print("\nBEFORE (original):")
print(df[quasi_ids].head(5).to_string())
print("\nAFTER (generalised):")
print(df_anon[quasi_ids].head(5).to_string())

AFTER generalisation:
k-anonymity: 1
Unique individuals (k=1): 53
Groups with k < 5: 101

BEFORE (original):
  applicant_info.gender applicant_info.date_of_birth applicant_info.zip_code
0                  Male                   2001-03-09                   10036
1                     M                   1992-03-31                   10032
2                  Male                   1989-10-24                   10075
3                  Male                   1983-04-25                   10077
4                     M                   1999-05-21                   10080

AFTER (generalised):
  applicant_info.gender  applicant_info.date_of_birth applicant_info.zip_code
0                     M                          2001                     100
1                     M                          1992                     100
2                     M                          1989                     100
3                     M                          1983                     100
4                

## 3.3 Differential Privacy

Add **calibrated noise** to aggregate outputs. Even if someone knows all other records, they can't figure out any single person's contribution.

Key parameter: **ε (epsilon)** — smaller ε means more noise, more privacy, but less accuracy.

In [51]:
import numpy as np

# differential privacy — add Laplace noise to the average income
income = pd.to_numeric(df['financials.annual_income'], errors='coerce').dropna()
true_mean = income.mean()

# sensitivity for the mean: (max - min) / n
sensitivity = (income.max() - income.min()) / len(income)

print(f"True average income: ${true_mean:,.2f}\n")

# try different epsilon values to see the privacy-accuracy trade-off
np.random.seed(42)
for eps in [0.1, 0.5, 1.0, 5.0]:
    noise = np.random.laplace(0, sensitivity / eps)
    print(f"  ε = {eps}  →  noisy mean = ${true_mean + noise:,.2f}  (noise: {noise:+.2f})")

print("\nSmaller epsilon = more noise = more privacy, less accuracy")

True average income: $82,705.10

  ε = 0.1  →  noisy mean = $81,711.06  (noise: -994.03)
  ε = 0.5  →  noisy mean = $84,299.47  (noise: +1594.38)
  ε = 1.0  →  noisy mean = $82,919.65  (noise: +214.56)
  ε = 5.0  →  noisy mean = $82,720.22  (noise: +15.12)

Smaller epsilon = more noise = more privacy, less accuracy


## 3.4 Our Recommendation

For this credit applications dataset, we would combine all three techniques:

1. **Pseudonymize** direct identifiers (name → token, email/SSN → salted hash, drop IP)
2. **Generalise** quasi-identifiers for k-anonymity (DOB → year, zip → 3 digits)
3. **Add Laplace noise** to any published aggregate statistics (e.g. average income)